Do DEGs analyiss again now with alpha=0.9 instead of HC k=10. use also absolute expressions not only relative

In [ ]:
# ── Cell 1 — Load and transfer niche labels ──────────────────────────

import os
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/DEGs")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'joint_alpha_0.9'

# Load niche labels
adata_niches = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

# Load RAW master file — this has true integer counts
adata_raw = sc.read_h5ad(
    "/path/to/data/"
    "Analyses_Erickson_Junyi/Thesis_Raw_Combined_Master.h5ad"
)

print("Niche adata shape:", adata_niches.shape)
print("Raw adata shape:", adata_raw.shape)
print("\nNiche barcodes (first 3):", adata_niches.obs_names[:3].tolist())
print("Raw barcodes (first 3):", adata_raw.obs_names[:3].tolist())

# Check overlap
overlap = adata_niches.obs_names.isin(adata_raw.obs_names).sum()
print(f"\nOverlapping barcodes: {overlap}")
print(f"Only in niches: {(~adata_niches.obs_names.isin(adata_raw.obs_names)).sum()}")
print(f"Only in raw: {(~adata_raw.obs_names.isin(adata_niches.obs_names)).sum()}")

In [ ]:
# Subset raw to only the 101,490 spots that have niche labels
adata_raw = adata_raw[adata_niches.obs_names].copy()
print("After subset:", adata_raw.shape)

# Verify raw counts are actually integers
X = adata_raw.X
sample_vals = X.data[:20] if sp.issparse(X) else X.flatten()[:20]
print("\nSample values (should be integers):", sample_vals)
print("Are integers:", np.all(sample_vals == sample_vals.astype(int)))

# Transfer niche labels
adata_raw.obs[CLUSTER_KEY] = adata_niches.obs[CLUSTER_KEY].values
print("\nNiche distribution:")
print(adata_raw.obs[CLUSTER_KEY].value_counts().sort_index())

In [ ]:
# Save raw counts before normalisation
adata_raw.layers['counts'] = adata_raw.X.copy()

# Normalise for DEG analysis
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# Verify
X_check = adata_raw.X
sample_check = X_check.data[:5] if sp.issparse(X_check) else X_check.flatten()[:5]
print("Post-normalisation sample values:", sample_check)
print("Max value:", adata_raw.X.max())

# Run DEGs
print("\nRunning DEGs — this may take 5-10 minutes...")
sc.tl.rank_genes_groups(
    adata_raw,
    groupby=CLUSTER_KEY,
    method='wilcoxon',
    key_added='rank_genes_joint_alpha_0.9',
    pts=True,
    n_genes=200
)
print("Done.")

In [ ]:
results = adata_raw.uns['rank_genes_joint_alpha_0.9']
niches = results['names'].dtype.names

all_degs = []
for niche in niches:
    df = pd.DataFrame({
        'gene':          results['names'][niche],
        'score':         results['scores'][niche],
        'pval':          results['pvals'][niche],
        'pval_adj':      results['pvals_adj'][niche],
        'logfoldchange': results['logfoldchanges'][niche],
    })
    df['niche'] = niche
    df = df[df['pval_adj'] < 0.05]
    df = df[df['logfoldchange'] > 0]
    all_degs.append(df)

degs_df = pd.concat(all_degs, ignore_index=True)
degs_df.to_csv(os.path.join(OUT_DIR, 'DEGs_joint_alpha_0.9.csv'), index=False)

print(f"Total significant upregulated DEGs: {len(degs_df)}")
print("\nTop 5 DEGs per niche:")
print(degs_df.groupby('niche').head(5)[
    ['niche', 'gene', 'logfoldchange', 'pval_adj']
].to_string(index=False))

In [ ]:
import scanpy as sc

adata_ref = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/reference_model/c2l_curated_reference.h5ad"
)

print("Shape:", adata_ref.shape)
print("\nobs columns:", adata_ref.obs.columns.tolist())
print("\nCell type column candidates:")
for col in adata_ref.obs.columns:
    n_unique = adata_ref.obs[col].nunique()
    if 2 <= n_unique <= 150:
        print(f"  {col}: {n_unique} unique values")
        print(f"    {adata_ref.obs[col].value_counts().head(5).to_dict()}")

print("\nvar columns:", adata_ref.var.columns.tolist())
print("\nX dtype:", adata_ref.X.dtype)
print("X max value:", adata_ref.X.max())
print("X min value:", adata_ref.X.min())
print("\nlayers:", list(adata_ref.layers.keys()) if adata_ref.layers else "none")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/DEGs")

# ── Load scRNA-seq reference ─────────────────────────────────────────
print("Loading scRNA-seq reference...")
adata_ref = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/reference_model/c2l_curated_reference.h5ad"
)
print("Shape:", adata_ref.shape)
print("Cell types:", adata_ref.obs['annotation_manual_fine'].value_counts().to_dict())

# ── Normalise for DEG analysis ───────────────────────────────────────
# X has raw counts — normalise in place
sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_ref)

# ── Run marker genes per cell type ───────────────────────────────────
print("\nRunning marker genes per cell type...")
sc.tl.rank_genes_groups(
    adata_ref,
    groupby='annotation_manual_fine',
    method='wilcoxon',
    key_added='rank_genes_celltype',
    pts=True,
    n_genes=100
)
print("Done.")

# ── Extract top markers per cell type ────────────────────────────────
results = adata_ref.uns['rank_genes_celltype']
cell_types = results['names'].dtype.names

scrna_markers = {}
all_scrna = []

for ct in cell_types:
    df = pd.DataFrame({
        'gene':          results['names'][ct],
        'logfoldchange': results['logfoldchanges'][ct],
        'pval_adj':      results['pvals_adj'][ct],
        'score':         results['scores'][ct],
    })
    df['cell_type'] = ct
    df = df[df['pval_adj'] < 0.05]
    df = df[df['logfoldchange'] > 0]
    scrna_markers[ct] = df
    all_scrna.append(df)

scrna_df = pd.concat(all_scrna, ignore_index=True)
scrna_df.to_csv(os.path.join(OUT_DIR, 'scrna_markers_per_celltype.csv'), 
                index=False)

print("\nTop 5 markers per cell type:")
print(scrna_df.groupby('cell_type').head(5)[
    ['cell_type', 'gene', 'logfoldchange', 'pval_adj']
].to_string(index=False))

The following answers: how many of the ground truth scrnaseq top 50 marker genes are in a particular niche

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Load your spatial DEGs ───────────────────────────────────────────
degs_df = pd.read_csv(os.path.join(OUT_DIR, 'DEGs_joint_alpha_0.9.csv'))

# ── For each niche, score against ALL cell types ─────────────────────
# For each cell type, compute how many of its top scRNA-seq markers
# appear in the spatial DEGs of each niche
# The cell type with the highest overlap score = identity of that niche

print("="*80)
print("NICHE IDENTITY — Unbiased scoring against scRNA-seq markers")
print("="*80)

# Get all cell types
all_cell_types = list(scrna_markers.keys())
all_niches = sorted(degs_df['niche'].unique())

# Build score matrix: niches x cell types
score_matrix = pd.DataFrame(
    index=all_niches,
    columns=all_cell_types,
    dtype=float
)

# Use top 50 scRNA-seq markers per cell type
n_markers = 50

for ct in all_cell_types:
    scrna_top = set(scrna_markers[ct].head(n_markers)['gene'].tolist())
    
    for niche in all_niches:
        spatial_genes = set(
            degs_df[degs_df['niche'] == niche]['gene'].tolist()
        )
        overlap = len(scrna_top & spatial_genes)
        # Normalise by number of spatial DEGs to avoid bias
        # toward niches with many DEGs
        # NEW — fraction of scRNA-seq signature recovered
        score = overlap / n_markers
        score_matrix.loc[niche, ct] = score

score_matrix = score_matrix.astype(float)

# ── Print top 3 matching cell types per niche ────────────────────────
print("\nTop 3 cell type matches per niche (by marker overlap score):")
print("Score = fraction of spatial DEGs that match scRNA-seq markers\n")

identity_rows = []
for niche in all_niches:
    top3 = score_matrix.loc[niche].nlargest(3)
    print(f"Niche {niche}:")
    for ct, score in top3.items():
        n_overlap = int(score * len(degs_df[degs_df['niche'] == niche]))
        overlap_genes = set(scrna_markers[ct].head(n_markers)['gene']) & \
                        set(degs_df[degs_df['niche'] == niche]['gene'])
        print(f"  {ct}: score={score:.4f} "
              f"({n_overlap} overlapping genes: "
              f"{', '.join(sorted(overlap_genes)[:5])}...)")
    
    identity_rows.append({
        'Niche': niche,
        'Top match': top3.index[0],
        'Top score': round(top3.iloc[0], 4),
        'Second match': top3.index[1],
        'Second score': round(top3.iloc[1], 4),
        'Third match': top3.index[2],
        'Third score': round(top3.iloc[2], 4),
    })
    print()

identity_df = pd.DataFrame(identity_rows)
print("\nSummary:")
print(identity_df.to_string(index=False))
identity_df.to_csv(
    os.path.join(OUT_DIR, 'niche_identity_from_scrna.csv'),
    index=False
)

# ── Heatmap of score matrix ──────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    score_matrix,
    cmap='YlOrRd',
    annot=True,
    fmt='.3f',
    ax=ax,
    linewidths=0.3,
    cbar_kws={'label': 'Overlap score\n(fraction of spatial DEGs matching scRNA-seq markers)'}
)
ax.set_title(
    'Niche identity scoring — spatial DEGs vs scRNA-seq markers\n'
    'Higher score = stronger match between spatial niche and cell type',
    fontsize=12
)
ax.set_xlabel('Cell type (scRNA-seq markers)')
ax.set_ylabel('Spatial niche (joint α=0.9)')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'niche_identity_heatmap_joint_graph.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: niche_identity_heatmap.png")

z score of celltype per cluster and output top 5 celltypes and their z score per cluster and get top 20 degs per cluster. and then name that (in a prostate context). also add screenshot of this plot. 

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scipy.sparse as sp
import os

OUT_DIR = ("/path/to/data/Cell2location/spatial_mapping/exploration_mean/DEGs")

CLUSTER_KEY = 'joint_alpha_0.9'

# ── Load data ────────────────────────────────────────────────────────
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)
adata_vis = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)
degs_df = pd.read_csv(os.path.join(OUT_DIR, 'DEGs_joint_alpha_0.9.csv'))

# ── Build proportion matrix ───────────────────────────────────────────
mean_ab = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_ab.columns = [c.replace('meanscell_abundance_w_sf_', '')
                   for c in mean_ab.columns]
props = mean_ab.div(mean_ab.sum(axis=1), axis=0)
props['niche'] = adata.obs[CLUSTER_KEY].values

degs_df['niche'] = degs_df['niche'].astype(str)

# Mean proportion per niche
niche_means = props.groupby('niche')[mean_ab.columns].mean()

# ── Z-score of cell type per niche ───────────────────────────────────
# Z-score across niches: for each cell type, how many SDs above/below
# the mean niche proportion is each niche
niche_zscore = (niche_means - niche_means.mean()) / niche_means.std()

print("Z-score matrix shape:", niche_zscore.shape)

# ── Top 5 cell types by z-score per niche ────────────────────────────
print("\n" + "="*80)
print("TOP 5 CELL TYPES BY Z-SCORE PER NICHE")
print("="*80)

summary_rows = []

for niche in sorted(niche_zscore.index, key=int):
    top5 = niche_zscore.loc[niche].nlargest(5)
    
    # Top 20 DEGs
    top20_degs = degs_df[degs_df['niche'] == str(niche)].head(20)['gene'].tolist()
    
    print(f"\nNiche {niche}:")
    print("  Top 5 cell types (z-score):")
    for ct, z in top5.items():
        raw_prop = niche_means.loc[niche, ct]
        print(f"    {ct}: z={z:.3f} (mean prop={raw_prop:.4f})")
    print(f"  Top 20 DEGs: {', '.join(top20_degs)}")
    
    summary_rows.append({
        'Niche': niche,
        'Top_CT_1': top5.index[0],
        'Z_1': round(top5.iloc[0], 3),
        'Top_CT_2': top5.index[1],
        'Z_2': round(top5.iloc[1], 3),
        'Top_CT_3': top5.index[2],
        'Z_3': round(top5.iloc[2], 3),
        'Top_CT_4': top5.index[3],
        'Z_4': round(top5.iloc[3], 3),
        'Top_CT_5': top5.index[4],
        'Z_5': round(top5.iloc[4], 3),
        'Top_20_DEGs': ', '.join(top20_degs),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT_DIR, 'niche_zscore_degs_joint_graph.csv'), index=False)
print("\nSaved: niche_zscore_degs.csv")

# ── Z-score heatmap ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    niche_zscore,
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt='.2f',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Z-score (cell type abundance per niche)'}
)
ax.set_title(
    'Cell type z-score per niche — HC Ward k=10\n'
    'Red = enriched relative to dataset mean | Blue = depleted',
    fontsize=12
)
ax.set_xlabel('Cell type')
ax.set_ylabel('Niche')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'niche_zscore_degs_joint_graph.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: niche_zscore_heatmap.png")

Check PCA3 expression:

In [ ]:
# Check PCA3 expression in niche 2
import scipy.sparse as sp
X_norm = adata_raw.X.toarray() if sp.issparse(adata_raw.X) else adata_raw.X
pca3_idx = adata_raw.var_names.get_loc('PCA3')
pca3_expr = pd.DataFrame({
    'PCA3': X_norm[:, pca3_idx],
    'niche': adata_raw.obs['joint_alpha_0.9'].values
})
print(pca3_expr.groupby('niche')['PCA3'].mean().sort_values(ascending=False))

In [ ]:
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

In [ ]:
mask = adata.obs['joint_alpha_0.9'] == '5'
print(f"n_spots: {mask.sum()}")
print("\nStudy:")
print(adata.obs[mask]['study'].value_counts())
print("\nTissue status:")
print(adata.obs[mask]['tissue_status'].value_counts())
print("\nPatient:")
print(adata.obs[mask]['patient'].value_counts())
print("\nSlide:")
print(adata.obs[mask]['slide_id'].value_counts())

see difference of qc per niche

In [ ]:
for niche in sorted(adata_raw.obs['joint_alpha_0.9'].unique()):
    mask = adata_raw.obs['joint_alpha_0.9'] == niche
    stats = adata_raw.obs.loc[mask, 
            ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].mean().round(1)
    print(f"Niche {niche}: n_genes={stats['n_genes_by_counts']:.0f}, "
          f"total_counts={stats['total_counts']:.0f}, "
          f"pct_mt={stats['pct_counts_mt']:.1f}%  "
          f"(n={mask.sum()})")

See difference of study enrichment per niche

In [ ]:
for niche in sorted(adata.obs['joint_alpha_0.9'].unique()):
    mask = adata.obs['joint_alpha_0.9'] == niche
    print(f"\n{'='*40}")
    print(f"Niche {niche} (n={mask.sum()})")
    print("Study:")
    print(adata.obs[mask]['study'].value_counts())
    print("Tissue status:")
    print(adata.obs[mask]['tissue_status'].value_counts())
    print("Patient (top 5):")
    print(adata.obs[mask]['patient'].value_counts().head(5))